In [46]:
import pandas as pd

# load dataset
df = pd.read_csv("soil_health_data.csv")

# show first rows
print(df.head())

# dataset info
print(df.info())

# check missing values
print(df.isnull().sum())

           State District                      Village    Block    Latitude  \
0  UTTAR PRADESH  Kasganj  Jakhaira Maheshpur - 215461  KASGANJ  27.8127295   
1  UTTAR PRADESH  Kasganj  Jakhaira Maheshpur - 215461  KASGANJ  27.8126326   
2  UTTAR PRADESH  Kasganj  Jakhaira Maheshpur - 215461  KASGANJ  27.8126112   
3  UTTAR PRADESH  Kasganj  Jakhaira Maheshpur - 215461  KASGANJ   27.812603   
4  UTTAR PRADESH  Kasganj  Jakhaira Maheshpur - 215461  KASGANJ  27.8125955   

    Longitude                                               Crop   pH     EC  \
0  78.5853472  ?? (All Variety), ??????? (All Variety), ??? (...  7.9   0.38   
1  78.5854108  ?? (All Variety), ????? (All Variety), ????? ?...  7.4  0.367   
2  78.5854667  ?? (All Variety), ????? (All Variety), ???????...  7.2  0.381   
3   78.585327  ?? (All Variety), ??????? (All Variety), ?????...  7.4  0.394   
4  78.5853649  ?? (All Variety), ??????? (All Variety), ?????...  7.9  0.393   

     OC    n      p      k      S     B    Z

In [47]:
# Select only soil nutrient columns
soil_df = df[['n','p','k','EC','OC','pH','S','Fe','Zn','Mn','Cu']]

# rename for consistency
soil_df = soil_df.rename(columns={
    'n':'N',
    'p':'P',
    'k':'K'
})

print(soil_df.head())
print(soil_df.info())

     N      P      K     EC    OC   pH      S    Fe    Zn    Mn    Cu
0  200      9  103.5   0.38  0.28  7.9  12.96  3.55  0.62  1.32   0.1
1  188  15.75   76.5  0.367  0.21  7.4   8.64  5.47  0.52   1.6  0.17
2  250  15.75    198  0.381  0.52  7.2  12.96  3.02  0.59  2.31  0.34
3  275  11.25    108  0.394  0.34  7.4  10.23  3.85  0.39   1.2  0.07
4  163      9  175.5  0.393  0.32  7.9  14.01  4.57  0.42  2.56   0.4
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5603 entries, 0 to 5602
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   N       5603 non-null   object
 1   P       5603 non-null   object
 2   K       5603 non-null   object
 3   EC      5603 non-null   object
 4   OC      5603 non-null   object
 5   pH      5603 non-null   object
 6   S       5603 non-null   object
 7   Fe      5603 non-null   object
 8   Zn      5603 non-null   object
 9   Mn      5603 non-null   object
 10  Cu      5603 non-null   object
dtype

In [48]:
soil_df = soil_df.apply(pd.to_numeric, errors='coerce')

print(soil_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5603 entries, 0 to 5602
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   N       5602 non-null   float64
 1   P       5602 non-null   float64
 2   K       5602 non-null   float64
 3   EC      5602 non-null   float64
 4   OC      5602 non-null   float64
 5   pH      5602 non-null   float64
 6   S       5602 non-null   float64
 7   Fe      5602 non-null   float64
 8   Zn      5602 non-null   float64
 9   Mn      5602 non-null   float64
 10  Cu      5602 non-null   float64
dtypes: float64(11)
memory usage: 481.6 KB
None


In [49]:
soil_df = soil_df.fillna(soil_df.mean())

print(soil_df.isnull().sum())

N     0
P     0
K     0
EC    0
OC    0
pH    0
S     0
Fe    0
Zn    0
Mn    0
Cu    0
dtype: int64


In [50]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

soil_scaled = scaler.fit_transform(soil_df)

soil_scaled = pd.DataFrame(soil_scaled, columns=soil_df.columns)

print(soil_scaled.head())

          N         P      K        EC        OC        pH         S  \
0  0.471698  0.065934  0.115  0.127090  0.085366  0.877778  0.119227   
1  0.443396  0.115385  0.085  0.122742  0.064024  0.822222  0.079485   
2  0.589623  0.115385  0.220  0.127425  0.158537  0.800000  0.119227   
3  0.648585  0.082418  0.120  0.131773  0.103659  0.822222  0.094112   
4  0.384434  0.065934  0.195  0.131438  0.097561  0.877778  0.128887   

         Fe        Zn        Mn        Cu  
0  0.265123  0.020000  0.005477  0.000699  
1  0.408514  0.016774  0.006639  0.001189  
2  0.225541  0.019032  0.009585  0.002378  
3  0.287528  0.012581  0.004979  0.000490  
4  0.341299  0.013548  0.010622  0.002797  


In [51]:
soil_scaled['SHI'] = (
    soil_scaled['N'] +
    soil_scaled['P'] +
    soil_scaled['K'] +
    soil_scaled['S'] +
    soil_scaled['OC'] +
    (1 - soil_scaled['EC']) * 1.2 +   # lower EC better
    (1 - abs(soil_scaled['pH'] - soil_scaled['pH'].median())) +
    soil_scaled['Fe'] +
    soil_scaled['Zn'] +
    soil_scaled['Mn'] +
    soil_scaled['Cu']
)/11

# convert to 0–100 scale
soil_scaled['SHI'] = soil_scaled['SHI'] * 100

print(soil_scaled[['SHI']].head())

         SHI
0  28.650656
1  29.654580
2  31.573313
3  30.773750
4  29.435574


In [52]:
X = soil_scaled[['N','P','K','EC','OC','pH','S','Fe','Zn','Mn','Cu']]

y = soil_scaled['SHI']

print(X.head())
print(y.head())
soil_scaled['SHI'].describe()

          N         P      K        EC        OC        pH         S  \
0  0.471698  0.065934  0.115  0.127090  0.085366  0.877778  0.119227   
1  0.443396  0.115385  0.085  0.122742  0.064024  0.822222  0.079485   
2  0.589623  0.115385  0.220  0.127425  0.158537  0.800000  0.119227   
3  0.648585  0.082418  0.120  0.131773  0.103659  0.822222  0.094112   
4  0.384434  0.065934  0.195  0.131438  0.097561  0.877778  0.128887   

         Fe        Zn        Mn        Cu  
0  0.265123  0.020000  0.005477  0.000699  
1  0.408514  0.016774  0.006639  0.001189  
2  0.225541  0.019032  0.009585  0.002378  
3  0.287528  0.012581  0.004979  0.000490  
4  0.341299  0.013548  0.010622  0.002797  
0    28.650656
1    29.654580
2    31.573313
3    30.773750
4    29.435574
Name: SHI, dtype: float64


count    5603.000000
mean       30.750392
std         2.896721
min        12.424242
25%        29.234279
50%        30.762087
75%        32.765377
max        42.629725
Name: SHI, dtype: float64

In [53]:
# Normalize SHI to full 0–100 range
shi_min = soil_scaled['SHI'].min()
shi_max = soil_scaled['SHI'].max()

soil_scaled['SHI'] = (soil_scaled['SHI'] - shi_min) / (shi_max - shi_min) * 100

print(soil_scaled['SHI'].describe())

count    5603.000000
mean       60.671600
std         9.590050
min         0.000000
25%        55.652270
50%        60.710319
75%        67.342523
max       100.000000
Name: SHI, dtype: float64


In [54]:
X = soil_scaled[['N','P','K','EC','OC','pH','S','Fe','Zn','Mn','Cu']]
y = soil_scaled['SHI']

In [55]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

Training size: (4482, 11)
Testing size: (1121, 11)


In [56]:
from sklearn.ensemble import RandomForestRegressor

# create model
soil_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

# train model
soil_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [57]:
from sklearn.metrics import r2_score, mean_absolute_error

# predictions
y_pred = soil_model.predict(X_test)

# evaluation
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("R2 Score:", r2)
print("Mean Absolute Error:", mae)

R2 Score: 0.9659881623059806
Mean Absolute Error: 0.44911539254879235


In [58]:
import joblib

joblib.dump(soil_model, "soil_health_model.pkl")

['soil_health_model.pkl']

In [59]:
import joblib
joblib.dump(scaler, "soil_scaler.pkl")

['soil_scaler.pkl']

In [60]:
def soil_category(score):
    if score < 40:
        return "Poor"
    elif score < 70:
        return "Moderate"
    else:
        return "Healthy"

soil_scaled["Soil_Category"] = soil_scaled["SHI"].apply(soil_category)

print(soil_scaled[["SHI","Soil_Category"]].head())

         SHI Soil_Category
0  53.720094      Moderate
1  57.043742      Moderate
2  63.396009      Moderate
3  60.748929      Moderate
4  56.318688      Moderate


In [61]:
import pandas as pd
def predict_soil_health(N, P, K, EC, OC, pH, S, Fe, Zn, Mn, Cu):

    # Create dataframe
    sample = pd.DataFrame({
        "N":[N],
        "P":[P],
        "K":[K],
        "EC":[EC],
        "OC":[OC],
        "pH":[pH],
        "S":[S],
        "Fe":[Fe],
        "Zn":[Zn],
        "Mn":[Mn],
        "Cu":[Cu]
    })

    # Scale input
    sample_scaled = scaler.transform(sample)
    sample_scaled = pd.DataFrame(sample_scaled, columns=sample.columns)

    # Predict SHI
    predicted_shi = soil_model.predict(sample_scaled)[0]

    # Category
    category = soil_category(predicted_shi)

    return predicted_shi, category


# ---------------------------------
# Example test
# ---------------------------------
shi, category = predict_soil_health(
    180, 12, 140, 0.35, 0.45, 7.0, 11, 4.2, 0.55, 2.0, 0.30
)

print("Soil Health Index:", round(shi,2))
print("Soil Category:", category)

Soil Health Index: 56.91
Soil Category: Moderate


In [62]:
shi, category = predict_soil_health(
    220,   # N
    14,    # P
    160,   # K
    0.30,  # EC
    0.50,  # OC
    7.1,   # pH
    12,    # S
    4.5,   # Fe
    0.60,  # Zn
    2.2,   # Mn
    0.30   # Cu
)
print("Soil Health Index:", round(shi,2))
print("Soil Category:", category)

Soil Health Index: 60.49
Soil Category: Moderate


In [64]:
shi, category = predict_soil_health(
    60,    # very low N
    3,     # very low P
    40,    # very low K
    0.75,  # very high EC (salinity)
    0.10,  # very low OC
    8.7,   # strongly alkaline
    2,     # very low S
    1.2,   # very low Fe
    0.10,  # very low Zn
    0.4,   # very low Mn
    0.02# Cu (very low copper)
)
print("Soil Health Index:", round(shi,2))
print("Soil Category:", category)

Soil Health Index: 36.12
Soil Category: Poor


In [65]:
shi, category = predict_soil_health(300,22,210,0.22,0.80,6.8,18,6.5,0.90,3.5,0.60)
print("Soil Health Index:", round(shi,2))
print("Soil Category:", category)

Soil Health Index: 71.21
Soil Category: Healthy
